In [3]:
import config as cfg
from data_acquirer import get_data, gen_data
from data_processor import process
import helpers
import heuristic
from heuristic import Path
from solver import solve_pdpcr

import copy
import datetime
from docplex.mp.model import Model
import logging
import heapq
import numpy as np
import time
import random 
import pickle

logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

In [4]:
#Li larger data

first_names = ['LC1']
sec_names = [2,4,6,8]
third_names = [i for i in range(1,5)]
data_paths =[f'lim/{j}00/{i}_{j}_{k}-5-uniform.csv'for i in first_names for j in sec_names for k in third_names]#
# num_robots = [10*i for i in range(,10)]
method = 5
file_type = 'csv'
is_depot_task = False

In [5]:
## Mass solve using tabu_search
# data_idcs = [16,18,19]
# data_paths = [f'generated{i}.csv' for i in data_idcs]
random.seed(cfg.RND_SEED)
np.random.seed(cfg.RND_SEED)
num_robots = [10,20]
file_type = 'csv'
is_depot_task = False
note = '3 neighborhoods'

for data_path in data_paths:
    for num_robot in num_robots:   
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            cfg.TABU_MAX_TL_ADD_SIZE = int(len(tasks)/6)
            cfg.TABU_MAX_TL_DROP_SIZE = int(len(tasks)/4)
            cfg.TABU_TL_REDISTRIBUTE_SIZE = int(len(tasks)/15)
            # cfg.TABU_NUM_REROUTE_NEIGHBORS =  int(len(tasks)/10)
            # cfg.TABU_NUM_REVERSE_NEIGHBORS =  int(len(tasks)/15)
            cfg.TABU_NUM_REROUTE_NEIGHBORS =  int(len(tasks)/8)
            cfg.TABU_NUM_REVERSE_NEIGHBORS =  int(len(tasks)/15)
            cfg.TABU_NUM_REDISTRIBUTE_NEIGHBORS = int(len(tasks)/15)
            # cfg.TABU_MAX_TL_ADD_SIZE = 12
            # cfg.TABU_MAX_TL_DROP_SIZE = 12
            # cfg.TABU_NUM_REROUTE_NEIGHBORS = 20
            # cfg.TABU_NUM_REVERSE_NEIGHBORS = 10
            cfg.TABU_MAX_ITER = 10
            cfg.TABU_FIND_FROM_ELITE_AFTER = 300 #iterations
            cfg.TABU_RESTART_AFTER = 600 #iterations
            cfg.TABU_REVERSE_ON_CRITICAL = True
            cfg.TABU_REROUTE_ON_CRITICAL = True
            cfg.TABU_REROUTE_LONGEST_EDGE = True
            cfg.TABU_REDISTRIBUTE_PROB = 0.3
            cfg.TABU_SAVE_FITNESSES = False
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            data_name = data_path.split('/')[2].split('.')[0]
            init_solution_path = f'solve_results/init_data/{data_name}_n_{num_robot}' #For fetch initial solution
            tik1 = time.time()  
            tabu = heuristic.TabuSearch(
                processed_h, 
                c_travel, 
                c_operate, 
                heuristic.SolutionFetcher, 
                init_solution_path = init_solution_path)
            best_sol, best_sol_obj, num_improve, avg_improve = tabu.search(num_robot,cfg.TABU_MAX_ITER,cfg.TABU_MAX_TL_ADD_SIZE)
            tok1 = time.time()
            tabu_time = tok1 - tik1
            print(f'Tabu time: {tabu_time}')
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                #add current time to data_name
                data_name += '_'+ datetime.datetime.now().strftime("%d-%m-%Y")
                tabu.save_fitness_progress(data_name)
        
            # solve_log = []
            # solve_log.append(data_path)
            # solve_log.append(num_robot)
            # solve_log.append(best_sol_obj)
            # solve_log.append(np.round(tabu_time,2))
            # solve_log.append(num_improve[0])
            # solve_log.append(num_improve[1])
            # solve_log.append(np.round(avg_improve[0],2))
            # solve_log.append(np.round(avg_improve[1],2))
            # now = datetime.datetime.now()
            # solve_log.append(now)
            # solve_log.append(note)
            # with open('solve_log_tabu.csv', 'a') as f:
            #     f.write(','.join([str(i) for i in solve_log])+'\n')

            # data_name = data_path.split('/')[2].split('.')[0]
            # pickle.dump(best_sol, open(f'solve_results/best_flows/{data_name}_n_{num_robot}.p', 'wb'))
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

04-May-26 22:22:41:INFO:Solving data lim/200/LC1_2_1-5-uniform.csv with 10 robots and 106 tasks


04-May-26 22:22:41:INFO:TABU SEARCH PARAMETERS
04-May-26 22:22:41:INFO:RANDOM_SEED: 16
04-May-26 22:22:41:INFO:NUM_REVERSE_NEIGHBORS: 7
04-May-26 22:22:41:INFO:NUM_REROUTE_NEIGHBORS: 13
04-May-26 22:22:41:INFO:NUM_REDISTRIBUTE_NEIGHBORS: 7
04-May-26 22:22:41:INFO:MAX_TL_ADD_SIZE: 17
04-May-26 22:22:41:INFO:MAX_TL_DROP_SIZE: 26
04-May-26 22:22:41:INFO:MAX_TL_REDISTRIBUTE_SIZE: 7
04-May-26 22:22:41:INFO:MAX_ITER: 10
04-May-26 22:22:41:INFO:FIND_FROM_ELITE_AFTER: 300
04-May-26 22:22:41:INFO:RESTART_AFTER: 600
04-May-26 22:22:41:INFO:REROUTE_ON_CRITICAL: True
04-May-26 22:22:41:INFO:REVERSE_ON_CRITICAL: True
04-May-26 22:22:41:INFO:REVERSE_CRITICAL_LONGEST_EDGE: True
04-May-26 22:22:41:INFO:Initial makespan: 921.3852276741839
04-May-26 22:22:41:INFO:Initial fitness: 921.3852276741839
04-May-26 22:22:41:INFO:Redistribute time 0.0020914077758789062
04-May-26 22:22:41:INFO:Redistribute time 0.0020737648010253906
04-May-26 22:22:41:INFO:Redistribute time 0.0031938552856445312
04-May-26 22:22:4

Tabu time: 1.1619980335235596


04-May-26 22:22:42:INFO:get neighbors time:
 reroute 0.0014223869030292218 
 reverse 0.004981790270124163 
 redistribute 0.00694690431867327
04-May-26 22:22:42:INFO:Iteration 0 done
04-May-26 22:22:42:INFO:Redistribute time 0.004444122314453125
04-May-26 22:22:42:INFO:Redistribute time 0.004520416259765625
04-May-26 22:22:42:INFO:Redistribute time 0.002624034881591797
04-May-26 22:22:42:INFO:Redistribute time 0.0031614303588867188
04-May-26 22:22:42:INFO:Redistribute time 0.0037851333618164062
04-May-26 22:22:42:INFO:Redistribute time 0.0023076534271240234
04-May-26 22:22:42:INFO:Redistribute time 0.004567384719848633
04-May-26 22:22:42:INFO:get neighbors time:
 reroute 0.0019034422360933744 
 reverse 0.007752690996442523 
 redistribute 0.0065446240561349055
04-May-26 22:22:42:INFO:Redistribute time 0.005467653274536133
04-May-26 22:22:42:INFO:Redistribute time 0.003577709197998047
04-May-26 22:22:42:INFO:Redistribute time 0.003105640411376953
04-May-26 22:22:42:INFO:Redistribute time 

Tabu time: 1.1950254440307617


04-May-26 22:22:43:INFO:Redistribute time 0.0030007362365722656
04-May-26 22:22:43:INFO:Redistribute time 0.0026175975799560547
04-May-26 22:22:43:INFO:Redistribute time 0.004132986068725586
04-May-26 22:22:43:INFO:Redistribute time 0.0043277740478515625
04-May-26 22:22:43:INFO:Redistribute time 0.0025031566619873047
04-May-26 22:22:43:INFO:Redistribute time 0.006672382354736328
04-May-26 22:22:43:INFO:get neighbors time:
 reroute 0.008957972893348107 
 reverse 0.004986218043736049 
 redistribute 0.006584746497017997
04-May-26 22:22:44:INFO:Redistribute time 0.0032591819763183594
04-May-26 22:22:44:INFO:Redistribute time 0.0009989738464355469
04-May-26 22:22:44:INFO:Redistribute time 0.0028057098388671875
04-May-26 22:22:44:INFO:Redistribute time 0.002306222915649414
04-May-26 22:22:44:INFO:Redistribute time 0.005515575408935547
04-May-26 22:22:44:INFO:Redistribute time 0.0022025108337402344
04-May-26 22:22:44:INFO:Redistribute time 0.002248048782348633
04-May-26 22:22:44:INFO:get neig

... (55 outputs left out)

04-May-26 22:27:15:INFO:TABU SEARCH PARAMETERS
04-May-26 22:27:15:INFO:RANDOM_SEED: 16
04-May-26 22:27:15:INFO:NUM_REVERSE_NEIGHBORS: 27
04-May-26 22:27:15:INFO:NUM_REROUTE_NEIGHBORS: 52
04-May-26 22:27:15:INFO:NUM_REDISTRIBUTE_NEIGHBORS: 27
04-May-26 22:27:15:INFO:MAX_TL_ADD_SIZE: 69
04-May-26 22:27:15:INFO:MAX_TL_DROP_SIZE: 104
04-May-26 22:27:15:INFO:MAX_TL_REDISTRIBUTE_SIZE: 27
04-May-26 22:27:15:INFO:MAX_ITER: 10
04-May-26 22:27:15:INFO:FIND_FROM_ELITE_AFTER: 300
04-May-26 22:27:15:INFO:RESTART_AFTER: 600
04-May-26 22:27:15:INFO:REROUTE_ON_CRITICAL: True
04-May-26 22:27:15:INFO:REVERSE_ON_CRITICAL: True
04-May-26 22:27:15:INFO:REVERSE_CRITICAL_LONGEST_EDGE: True
04-May-26 22:27:15:INFO:Initial makespan: 6440.16595824007
04-May-26 22:27:15:INFO:Initial fitness: 6440.16595824007
04-May-26 22:27:16:INFO:Redistribute time 0.03196454048156738
04-May-26 22:27:16:INFO:Redistribute time 0.0066568851470947266
04-May-26 22:27:16:INFO:Redistribute time 0.008042097091674805
04-May-26 22:27:16

Tabu time: 20.477283716201782


04-May-26 22:27:36:INFO:TABU SEARCH PARAMETERS
04-May-26 22:27:36:INFO:RANDOM_SEED: 16
04-May-26 22:27:36:INFO:NUM_REVERSE_NEIGHBORS: 27
04-May-26 22:27:36:INFO:NUM_REROUTE_NEIGHBORS: 52
04-May-26 22:27:36:INFO:NUM_REDISTRIBUTE_NEIGHBORS: 27
04-May-26 22:27:36:INFO:MAX_TL_ADD_SIZE: 69
04-May-26 22:27:36:INFO:MAX_TL_DROP_SIZE: 104
04-May-26 22:27:36:INFO:MAX_TL_REDISTRIBUTE_SIZE: 27
04-May-26 22:27:36:INFO:MAX_ITER: 10
04-May-26 22:27:36:INFO:FIND_FROM_ELITE_AFTER: 300
04-May-26 22:27:36:INFO:RESTART_AFTER: 600
04-May-26 22:27:36:INFO:REROUTE_ON_CRITICAL: True
04-May-26 22:27:36:INFO:REVERSE_ON_CRITICAL: True
04-May-26 22:27:36:INFO:REVERSE_CRITICAL_LONGEST_EDGE: True
04-May-26 22:27:36:INFO:Initial makespan: 3396.8532917301677
04-May-26 22:27:37:INFO:Initial fitness: 3396.8532917301677
04-May-26 22:27:37:INFO:Redistribute time 0.02846503257751465
04-May-26 22:27:37:INFO:Redistribute time 0.026667356491088867
04-May-26 22:27:37:INFO:Redistribute time 0.04383683204650879
04-May-26 22:27:

Tabu time: 19.5467050075531


In [3]:
## Mass solve using tabu_search
# data_idcs = [16,18,19]
# data_paths = [f'generated{i}.csv' for i in data_idcs]

random.seed(cfg.RND_SEED)
np.random.seed(cfg.RND_SEED)
num_robots = [5]
file_type = 'csv'
is_depot_task = False
note = '2 neighborhoods'

for data_path in data_paths:
    for num_robot in num_robots:   
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            cfg.TABU_MAX_TL_ADD_SIZE = int(len(tasks)/8)
            cfg.TABU_MAX_TL_DROP_SIZE = int(len(tasks)/4)
            cfg.TABU_TL_REDISTRIBUTE_SIZE = int(len(tasks)/15)
            cfg.TABU_NUM_REROUTE_NEIGHBORS =  int(len(tasks)/3)
            cfg.TABU_NUM_REVERSE_NEIGHBORS =  int(len(tasks)/9)
            cfg.TABU_NUM_REDISTRIBUTE_NEIGHBORS =0# int(len(tasks)/15)
            # cfg.TABU_MAX_TL_ADD_SIZE = 12
            # cfg.TABU_MAX_TL_DROP_SIZE = 12
            # cfg.TABU_NUM_REROUTE_NEIGHBORS = 20
            # cfg.TABU_NUM_REVERSE_NEIGHBORS = 10
            cfg.TABU_MAX_ITER = 5000
            cfg.TABU_FIND_FROM_ELITE_AFTER = 300 #iterations
            cfg.TABU_RESTART_AFTER = 600 #iterations
            cfg.TABU_REVERSE_ON_CRITICAL = True
            cfg.TABU_REROUTE_ON_CRITICAL = True
            cfg.TABU_REROUTE_LONGEST_EDGE = True
            cfg.TABU_REDISTRIBUTE_PROB = 0.3
            cfg.TABU_SAVE_FITNESSES = True
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            data_name = data_path.split('/')[2].split('.')[0]
            init_solution_path = f'solve_results/init_data/{data_name}_n_{num_robot}' #For fetch initial solution
            tik1 = time.time()  
            tabu = heuristic.TabuSearch(
                processed_h, 
                c_travel, 
                c_operate, 
                heuristic.SolutionFetcher, 
                init_solution_path = init_solution_path)
            best_sol, best_sol_obj, num_improve, avg_improve = tabu.search(num_robot,cfg.TABU_MAX_ITER,cfg.TABU_MAX_TL_ADD_SIZE)
            tok1 = time.time()
            tabu_time = tok1 - tik1
            print(f'Tabu time: {tabu_time}')
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                #add current time to data_name
                data_name += '_'+ datetime.datetime.now().strftime("%d-%m-%Y")
                tabu.save_fitness_progress(data_name)
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(best_sol_obj)
            solve_log.append(np.round(tabu_time,2))
            solve_log.append(num_improve[0])
            solve_log.append(num_improve[1])
            solve_log.append(np.round(avg_improve[0],2))
            solve_log.append(np.round(avg_improve[1],2))
            now = datetime.datetime.now()
            solve_log.append(now)
            solve_log.append(note)
            with open('solve_log_tabu.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')

            pickle.dump(best_sol, open(f'solve_results/best_flows/{data_name}_n_{num_robot}.p', 'wb'))

        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            # raise e
            print(e)

30-Apr-26 10:59:34:INFO:Solving data lim/600/LC1_6_1-5-uniform.csv with 5 robots and 315 tasks


30-Apr-26 10:59:35:INFO:TABU SEARCH PARAMETERS
30-Apr-26 10:59:35:INFO:RANDOM_SEED: 16
30-Apr-26 10:59:35:INFO:NUM_REVERSE_NEIGHBORS: 35
30-Apr-26 10:59:35:INFO:NUM_REROUTE_NEIGHBORS: 105
30-Apr-26 10:59:35:INFO:NUM_REDISTRIBUTE_NEIGHBORS: 0
30-Apr-26 10:59:35:INFO:MAX_TL_ADD_SIZE: 39
30-Apr-26 10:59:35:INFO:MAX_TL_DROP_SIZE: 78
30-Apr-26 10:59:35:INFO:MAX_TL_REDISTRIBUTE_SIZE: 21
30-Apr-26 10:59:35:INFO:MAX_ITER: 5000
30-Apr-26 10:59:35:INFO:FIND_FROM_ELITE_AFTER: 300
30-Apr-26 10:59:35:INFO:RESTART_AFTER: 600
30-Apr-26 10:59:35:INFO:REROUTE_ON_CRITICAL: True
30-Apr-26 10:59:35:INFO:REVERSE_ON_CRITICAL: True
30-Apr-26 10:59:35:INFO:REVERSE_CRITICAL_LONGEST_EDGE: True
30-Apr-26 10:59:35:INFO:Initial makespan: 5556.900243471716
30-Apr-26 10:59:35:INFO:Initial fitness: 5556.900243471716
30-Apr-26 10:59:36:INFO:Best fitness updated: 5547.589505700797, chosen neighborhood: 0
30-Apr-26 10:59:36:INFO:Iteration 0 done
30-Apr-26 10:59:36:INFO:Best fitness updated: 5542.5664869612565, chosen ne

Viet generalized sol is 5842.91
Viet generalized sol is 6152.93
Viet generalized sol is 6151.3
Viet generalized sol is 5692.36
Viet generalized sol is 5530.06
Viet generalized sol is 5973.52
Viet generalized sol is 5983.89
Viet generalized sol is 5666.13
Viet generalized sol is 5964.17
Viet generalized sol is 5705.1
Viet generalized sol is 5713.68
Viet generalized sol is 5666.13
Viet generalized sol is 5917.75
Viet generalized sol is 5701.34
Viet generalized sol is 5880.06
Viet generalized sol is 5912.83
Viet generalized sol is 5590.88
Viet generalized sol is 6027.35
Viet generalized sol is 6012.97
Viet generalized sol is 5961.52
Viet generalized sol is 5874.79
Viet generalized sol is 5849.56
Viet generalized sol is 5811.35
Viet generalized sol is 5615.03
Viet generalized sol is 5785.35
Viet generalized sol is 6072.7
Viet generalized sol is 5835.08
Viet generalized sol is 6114.42
Viet generalized sol is 5765.73
Viet generalized sol is 5869.78


30-Apr-26 11:41:25:INFO:Iteration 2800 done
30-Apr-26 11:42:08:INFO:Iteration 2850 done
30-Apr-26 11:42:51:INFO:Iteration 2900 done
30-Apr-26 11:43:34:INFO:Iteration 2950 done
30-Apr-26 11:44:23:INFO:Iteration 3000 done
30-Apr-26 11:45:05:INFO:Iteration 3050 done
30-Apr-26 11:45:29:INFO:No update for 300 iterations, re-search from best solution
30-Apr-26 11:45:48:INFO:Iteration 3100 done
30-Apr-26 11:46:31:INFO:Iteration 3150 done
30-Apr-26 11:47:15:INFO:Iteration 3200 done
30-Apr-26 11:47:59:INFO:Iteration 3250 done
30-Apr-26 11:48:43:INFO:Iteration 3300 done
30-Apr-26 11:49:26:INFO:Iteration 3350 done
30-Apr-26 11:49:50:INFO:No update for 600 iterations, restarting from initial solution


Viet generalized sol is 5711.56
Viet generalized sol is 5778.33
Viet generalized sol is 5960.68
Viet generalized sol is 6031.55
Viet generalized sol is 5831.48
Viet generalized sol is 6001.86
Viet generalized sol is 5668.25
Viet generalized sol is 5891.11
Viet generalized sol is 5924.93
Viet generalized sol is 5809.97
Viet generalized sol is 5660.17
Viet generalized sol is 6007.18
Viet generalized sol is 5719.36
Viet generalized sol is 5899.6
Viet generalized sol is 5753.4
Viet generalized sol is 5969.72
Viet generalized sol is 5734.22
Viet generalized sol is 6125.56
Viet generalized sol is 5880.13
Viet generalized sol is 6004.43
Viet generalized sol is 5822.03
Viet generalized sol is 5808.19
Viet generalized sol is 6105.19
Viet generalized sol is 5666.13
Viet generalized sol is 6008.95
Viet generalized sol is 5733.07
Viet generalized sol is 6117.05
Viet generalized sol is 6192.36
Viet generalized sol is 6028.23
Viet generalized sol is 5728.52


30-Apr-26 11:50:32:INFO:Iteration 3400 done
30-Apr-26 11:51:16:INFO:Iteration 3450 done
30-Apr-26 11:51:59:INFO:Iteration 3500 done
30-Apr-26 11:52:43:INFO:Iteration 3550 done
30-Apr-26 11:53:27:INFO:Iteration 3600 done
30-Apr-26 11:54:15:INFO:Iteration 3650 done
30-Apr-26 11:54:40:INFO:No update for 300 iterations, re-search from best solution
30-Apr-26 11:54:59:INFO:Iteration 3700 done
30-Apr-26 11:55:43:INFO:Iteration 3750 done
30-Apr-26 11:56:26:INFO:Iteration 3800 done
30-Apr-26 11:57:09:INFO:Iteration 3850 done
30-Apr-26 11:57:54:INFO:Iteration 3900 done
30-Apr-26 11:58:36:INFO:Iteration 3950 done
30-Apr-26 11:59:01:INFO:No update for 600 iterations, restarting from initial solution


... (27 outputs left out)

30-Apr-26 17:35:43:INFO:Iteration 4350 done
30-Apr-26 17:37:19:INFO:Iteration 4400 done
30-Apr-26 17:38:54:INFO:Iteration 4450 done
30-Apr-26 17:40:31:INFO:Iteration 4500 done
30-Apr-26 17:42:07:INFO:Iteration 4550 done
30-Apr-26 17:43:43:INFO:Iteration 4600 done
30-Apr-26 17:44:02:INFO:No update for 300 iterations, re-search from best solution
30-Apr-26 17:45:25:INFO:Iteration 4650 done
30-Apr-26 17:47:06:INFO:Iteration 4700 done
30-Apr-26 17:48:46:INFO:Iteration 4750 done
30-Apr-26 17:50:28:INFO:Iteration 4800 done
30-Apr-26 17:52:06:INFO:Iteration 4850 done
30-Apr-26 17:53:44:INFO:Iteration 4900 done
30-Apr-26 17:54:01:INFO:No update for 600 iterations, restarting from initial solution


Viet generalized sol is 10999.06
Viet generalized sol is 10616.73
Viet generalized sol is 10735.89
Viet generalized sol is 10915.69
Viet generalized sol is 10693.87
Viet generalized sol is 10057.18
Viet generalized sol is 10615.75
Viet generalized sol is 10608.2
Viet generalized sol is 10691.07
Viet generalized sol is 10667.43
Viet generalized sol is 10317.14
Viet generalized sol is 10641.86
Viet generalized sol is 10651.72
Viet generalized sol is 10755.23
Viet generalized sol is 10495.76
Viet generalized sol is 10715.71
Viet generalized sol is 10237.48
Viet generalized sol is 10945.37
Viet generalized sol is 10616.73
Viet generalized sol is 10475.81
Viet generalized sol is 10303.94
Viet generalized sol is 10288.13
Viet generalized sol is 11115.16
Viet generalized sol is 10367.9
Viet generalized sol is 10191.38
Viet generalized sol is 10145.33
Viet generalized sol is 10061.27
Viet generalized sol is 11162.4
Viet generalized sol is 10834.07
Viet generalized sol is 10729.03


30-Apr-26 17:56:13:INFO:Iteration 4950 done
30-Apr-26 17:57:47:INFO:Best solution found: 9686.32418828859


Tabu time: 8401.999512195587


In [ ]:
## Mass solve using tabu_search
#ONLY REDISTRIBUTE
num_robots = [5]
file_type = 'csv'
is_depot_task = False

for num_robot in num_robots:
    for data_path in data_paths:
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            cfg.TABU_MAX_TL_ADD_SIZE = int(len(tasks)/8)
            cfg.TABU_MAX_TL_DROP_SIZE = int(len(tasks)/8)
            cfg.TABU_NUM_REROUTE_NEIGHBORS =  0 #int(len(tasks)/5)
            cfg.TABU_NUM_REVERSE_NEIGHBORS = 0 # int(len(tasks)/10)
            cfg.TABU_NUM_REDISTRIBUTE_NEIGHBORS = int(len(tasks)/20)
            # cfg.TABU_MAX_TL_ADD_SIZE = 12
            # cfg.TABU_MAX_TL_DROP_SIZE = 12
            # cfg.TABU_NUM_REROUTE_NEIGHBORS = 20
            cfg.TABU_MAX_ITER = 200
            cfg.TABU_FIND_FROM_ELITE_AFTER = 200 #iterations
            cfg.TABU_RESTART_AFTER = 400 #iterations
            cfg.TABU_REVERSE_ON_CRITICAL = True
            cfg.TABU_REROUTE_ON_CRITICAL = True
            cfg.TABU_REROUTE_LONGEST_EDGE = True
            cfg.TABU_REDISTRIBUTE_PROB = 0.3
            cfg.TABU_SAVE_FITNESSES = True
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            data_name = data_path.split('/')[2].split('.')[0]
            init_solution_path = f'solve_results/init_data/{data_name}' #For fetch initial solution
            tik1 = time.time()
            tabu = heuristic.TabuSearch(
                processed_h, 
                c_travel, 
                c_operate, 
                heuristic.SolutionFetcher, 
                init_solution_path = init_solution_path)
            best_sol, best_sol_obj = tabu.search(num_robot,cfg.TABU_MAX_ITER)
            tok1 = time.time()
            tabu_time = tok1 - tik1
            print(f'Tabu time: {tabu_time}')
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                #add current time to data_name
                data_name += '_'+ datetime.datetime.now().strftime("%Y-%m-%d") + 'redis_only'
                tabu.save_fitness_progress(data_name)
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(best_sol_obj)
            solve_log.append(np.round(tabu_time,2))
            now = datetime.datetime.now()
            solve_log.append(now)
            with open('solve_log_tabu.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
## Mass solve using tabu_search
#ONLY REROUTE
num_robots = [10]
file_type = 'csv'
is_depot_task = False

for num_robot in num_robots:
    for data_path in data_paths:
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            cfg.TABU_MAX_TL_ADD_SIZE = int(len(tasks)/8)
            cfg.TABU_MAX_TL_DROP_SIZE = int(len(tasks)/8)
            cfg.TABU_NUM_REROUTE_NEIGHBORS =  int(len(tasks)/7)
            cfg.TABU_NUM_REVERSE_NEIGHBORS = 0 # int(len(tasks)/10)
            cfg.TABU_NUM_REDISTRIBUTE_NEIGHBORS = 0 #int(len(tasks)/20)
            # cfg.TABU_MAX_TL_ADD_SIZE = 12
            # cfg.TABU_MAX_TL_DROP_SIZE = 12
            # cfg.TABU_NUM_REROUTE_NEIGHBORS = 20
            # cfg.TABU_NUM_REVERSE_NEIGHBORS = 10
            cfg.TABU_MAX_ITER = 200
            cfg.TABU_FIND_FROM_ELITE_AFTER = 200 #iterations
            cfg.TABU_RESTART_AFTER = 400 #iterations
            cfg.TABU_REVERSE_ON_CRITICAL = True
            cfg.TABU_REROUTE_ON_CRITICAL = True
            cfg.TABU_REROUTE_LONGEST_EDGE = True
            cfg.TABU_REDISTRIBUTE_PROB = 0.3
            cfg.TABU_SAVE_FITNESSES = True
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            data_name = data_path.split('/')[2].split('.')[0]
            init_solution_path = f'solve_results/init_data/{data_name}' #For fetch initial solution
            tik1 = time.time()
            tabu = heuristic.TabuSearch(
                processed_h, 
                c_travel, 
                c_operate, 
                heuristic.SolutionFetcher, 
                init_solution_path = init_solution_path)
            
            best_sol, best_sol_obj = tabu.search(num_robot,cfg.TABU_MAX_ITER)
            tok1 = time.time()
            tabu_time = tok1 - tik1
            print(f'Tabu time: {tabu_time}')
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                #add current time to data_name
                data_name += '_'+ datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + 'reroute_only'
                tabu.save_fitness_progress(data_name)
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(best_sol_obj)
            solve_log.append(np.round(tabu_time,2))
            now = datetime.datetime.now()
            solve_log.append(now)
            with open('solve_log_tabu.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
cProfile.run(tabu.search(num_robot,cfg.TABU_MAX_ITER))

In [ ]:
## mass solve using heuristics
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            tik1 = time.time()
            solve_result_h = heuristic.solve_greedy_1(processed_h, num_robot)
            tok1 = time.time()
            greedy1_time = tok1 - tik1
            print(f'Greedy 1 time: {greedy1_time}')
            tik2 = time.time()
            solve_result_h_jp = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_jp_time = tok2 -tik2
            print(f'Greedy jp time: {greedy_jp_time}')
            tik3 = time.time()
            solve_result_h_jp_2 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3
            print(f'Greedy jp time 2: {greedy_jp_time_2}')

        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(solve_result_h.get_makespan().round(2))
            solve_log.append(np.round(greedy1_time,2))
            solve_log.append(solve_result_h_jp.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time,2))
            solve_log.append(solve_result_h_jp_2.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            with open('solve_log_h_2.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
import helpers
## Mass solve 2 opt
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_h = heuristic.process_greedy_1(raw)

            tik2 = time.time()
            solve_result_h2 = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_h2 = tok2 -tik2

            tik3 = time.time()
            solve_result_h3 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3

            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)
            tik = time.time()
            solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)
            graph,makespan_2opt = solve_opt.solve(total_robots= num_robot)
            tok = time.time()
            two_opt_time = tok - tik

            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_result_h2.get_makespan().round(2))
            solve_log.append(np.round(greedy_h2,2))
            solve_log.append(solve_result_h3.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            solve_log.append(makespan_2opt.round(2))
            solve_log.append(np.round(two_opt_time,2))
            now = datetime.datetime.now()
            solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))
            with open('solve_log_h_2opt.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
## Mass solve Viet 

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')
consider_process_time =True
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)
            tik = time.time()
            solve_viet = heuristic.VietSolverOrig(processed_h, c_travel, c_operate)
            solve_viet.solve(num_robot,consider_process_time)
            tok = time.time()
            time_viet = tok - tik
            print(f'makespan: {solve_viet.get_makespan()}')
            print('Done')

            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_viet.get_makespan().round(2))
            solve_log.append(np.round(time_viet,2))
            with open('solve_log_h_Viet_09-04-process-time.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)


In [ ]:
## Mass solve Viet k tasks

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

tasks_each_iters = [5]
consider_process_time =False
for task_each_ite in tasks_each_iters:
    for data_path in data_paths:
        for num_robot in num_robots:
            try:
                print(f'Solving for {data_path} with {num_robot} robots')
                raw = get_data(data_path, file_type, is_depot_task)
                processed_h = heuristic.process_greedy_1(raw)
                depot = processed_h.depot
                tasks = processed_h.tasks
                c_travel = helpers.create_task_dist_dict(depot,tasks)
                c_operate = helpers.create_operate_dict(depot,tasks)
                tik = time.time()
                solve_viet = heuristic.VGeneralizedSolver(processed_h, c_travel, c_operate)
                sol = solve_viet.solve(num_robot,task_each_ite,consider_process_time)
                tok = time.time()
                time_viet = tok - tik
                makespan = sol.get_makespan()
                print(f'makespan: {makespan}')
                print('Done')

                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)
                solve_log.append(task_each_ite)
                solve_log.append(makespan.round(2))
                solve_log.append(np.round(time_viet,2))
                with open('solve_log_h_Viet k task large instances 23-04.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)


In [ ]:
a ={0: 0.0, 1: 599.9817763899168, 2: 271.90904527065663, 3: 283.1551518161024, 4: 142.61641735789044, 5: 239.53321272842314, 6: 503.8917411508151, 7: 320.3896980241406, 8: 347.00099826945745, 9: 177.80785134520917, 10: 194.39276118209756, -1: 0}

In [ ]:
#Li larger data

first_names = ['LC1']
sec_names = [2,4,6,8,10]
third_names = [i for i in range(1,11)]
data_paths =[f'lim/{j}00/{i}_{j}_{k}-5-uniform.csv'for i in first_names for j in sec_names for k in third_names]#
num_robots = [10*i for i in range(1,4)]
method = 5
file_type = 'csv'
is_depot_task = False

In [ ]:
## Mass solve Viet or jp k tasks with random k

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')
num_robots = [5,10,20]
ite = 50
# tasks_each_iter_ranges = [[2,4]]
note = 'JP_solver 80 15 5'
tasks_each_iter_ranges = [[1,2]]
consider_process_time =False
for task_each_ite in tasks_each_iter_ranges:
    for data_path in data_paths:
        for num_robot in num_robots:
            try:
                print(f'Solving for {data_path} with {num_robot} robots')
                raw = get_data(data_path, file_type, is_depot_task)
                processed_h = heuristic.process_greedy_1(raw)
                depot = processed_h.depot
                tasks = processed_h.tasks
                c_travel = helpers.create_task_dist_dict(depot,tasks)
                c_operate = helpers.create_operate_dict(depot,tasks)

                solve_result_h_jp = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
                TT_makespan = solve_result_h_jp.get_makespan()

                tik = time.time()
                cur_best = np.infty
                cur_worst = -np.infty
                makespan_results = []
                no_better_TT = 0

                for i in range(ite):
                    
                    solve_viet = heuristic.JPGeneralizedSolver(processed_h, c_travel, c_operate)
                    # solve_viet = heuristic.VGeneralizedSolver(processed_h, c_travel, c_operate)
                    sol = solve_viet.solve(num_robot,'random',consider_process_time,task_range=task_each_ite)
                    makespan = sol.get_makespan()
                    makespan_results.append(makespan)
                    if makespan < cur_best:
                        cur_best = makespan
                    if makespan > cur_worst:
                        cur_worst = makespan
                    if makespan <= TT_makespan:
                        no_better_TT +=1
                avg_makespan = np.mean(makespan_results)
                tok = time.time()
                cpu = tok - tik
            
                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)
                solve_log.append(ite)
                solve_log.append(cur_best.round(2))
                solve_log.append(cur_worst.round(2))
                solve_log.append(avg_makespan.round(2))
                solve_log.append(np.round(1-cur_best/TT_makespan,4)*100)
                solve_log.append(np.round(1-avg_makespan/TT_makespan,4)*100)
                solve_log.append(np.round(cpu,2))
                solve_log.append(no_better_TT)
                solve_log.append(note)
                with open('solve_log_h_JP large instances these.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)


In [ ]:
raw = get_data(data_path, file_type, is_depot_task)
processed_h = heuristic.process_greedy_1(raw)
depot = processed_h.depot
tasks = processed_h.tasks
c_travel = helpers.create_task_dist_dict(depot,tasks)
c_operate = helpers.create_operate_dict(depot,tasks)
tik = time.time()
solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)

In [ ]:
#Get data

#Read file solve_log.csv, save it as a dataframe
import pandas as pd
solve_cplex_df = pd.read_csv('solve_log.csv', header=None)
solve_cplex_df.columns = ['data_path', 'num_robot', 'solve_time', 'objective_value', 'processed_nodes', 'gap', 'solve_date']

solve_h_df = pd.read_csv('solve_log_h.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_jp', 'time_greedy_jp']

In [ ]:
# right join 2 dataframes on column data_path
merged_df = pd.merge(solve_cplex_df, solve_h_df, on='data_path', how='right')

In [ ]:
merged_df['diff_h_1'] = np.round(1- merged_df['objective_value_1']/merged_df['objective_value'],2)
merged_df['diff_h_jp'] = np.round(1- merged_df['objective_value_jp']/merged_df['objective_value'],2)
#change order of columns in meged_df as follow: data_path, num_robot, objective_value, solve_time, objective_value_1, time_greedy_1, diff_h_1, objective_value_jp, time_greedy_jp, diff_h_jp,
merged_df = merged_df[['data_path', 'num_robot', 'objective_value', 'solve_time', 'objective_value_1', 'time_greedy_1', 'diff_h_1', 'objective_value_jp', 'time_greedy_jp', 'diff_h_jp']]
merged_df
#Remove the 'lim' prefix from data_path
merged_df['data_path'] = merged_df['data_path'].str.replace('lim/','')
#round solve_time to 2 decimal places
merged_df['solve_time'] = np.round(merged_df['solve_time'],2)
#round objective_value to 2 decimal places
merged_df['objective_value'] = np.round(merged_df['objective_value'],2)

In [ ]:
merged_df.to_latex('merged_df.tex', index=True)

In [ ]:
#Temp
import pandas as pd
solve_h_df = pd.read_csv('solve_log_h_2.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3']
#Keep the following columns: data_path, objective_value_2, time_greedy_2
solve_h_df = solve_h_df[['data_path', 'objective_value_2', 'time_greedy_2']]

In [ ]:
solve_h_df_2opt = pd.read_csv('solve_log_h_2opt.csv', header=None)
solve_h_df_2opt.columns = ['data_path', 'num_robot', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']

In [ ]:
#Join 2 dataframes on data_path
merged_df_2 = pd.merge(solve_h_df, solve_h_df_2opt, on='data_path', how='right')
#Change the order of columns as follow: data_path, num_robot, objective_value_2, time_greedy_2, objective_value_3, time_greedy_3, objective_value_2opt, time_2opt
merged_df_2 = merged_df_2[['data_path', 'num_robot', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']]

In [ ]:
#save the data as a latex table
merged_df_2.to_latex('merged_df_2.tex', index=True)